# Этап 8. Правило решения, сравнения и план объема (письмо 29.10, п. 6, 7, 9)

Код: `rssrl/stats.py` (правило решения, перекрестный бутстреп, точная граница, Холм, β*), `rssrl/volume.py` (бюджет, оценка полуширины по пробным политикам), `rssrl/training.py` (целевая среда обучения). Проверки: `tests/test_stats.py`, `tests/test_volume.py`.

Здесь нет окончательного объема: его считаем после 2–3 пробных политик PPO на целевой среде. Пилот с IDLE не оценивает разброс будущих политик и не годится для расчета объема.

In [1]:
import sys
sys.path.insert(0, '..')
import json
from pathlib import Path
import numpy as np
import pandas as pd
from rssrl.stats import COMPARISONS, beta_star, compare, decide, holm, zero_upper_bound
from rssrl.volume import budget

RES = Path('..') / 'results'
PILOT = RES / 'pilot_08'
PILOT.mkdir(parents=True, exist_ok=True)

## 1. Правило решения: две отдельные характеристики

Для разности частот столкновений Δ = P(A) − P(B) с ДИ:
1. **Статистический знак**: «хуже» — весь ДИ > 0; «лучше» — весь ДИ < 0; иначе «не различимо».
2. **Практическая величина** относительно допуска ±2 п.п.: «в пределах допуска» — весь ДИ внутри [−2; 2] п.п.; «за пределами допуска» — весь ДИ по одну сторону вне него; «не определена» — ДИ пересекает границу допуска.

Каждый ДИ получает ровно одну пару меток, приоритеты не нужны.

In [2]:
examples = [(0.005, 0.015), (0.03, 0.06), (0.01, 0.03), (-0.01, 0.01), (-0.05, 0.05), (-0.06, -0.03)]
pd.DataFrame([{'ДИ, п.п.': f'[{100*lo:g}; {100*hi:g}]', 'знак': decide((lo, hi))[0],
               'величина': decide((lo, hi))[1]} for lo, hi in examples])

,"ДИ, п.п.",знак,величина
0,[0.5; 1.5],хуже,в пределах допуска
1,[3; 6],хуже,за пределами допуска
2,[1; 3],хуже,не определена
3,[-1; 1],не различимо,в пределах допуска
4,[-5; 5],не различимо,не определена
5,[-6; -3],лучше,за пределами допуска


Пример из письма: [0,5; 1,5] п.п. — «хуже, в пределах допуска»: увеличение риска статистически обнаружено, но меньше выбранного практического порога.

## 2. Основные сравнения и семьи поправки Холма

Обозначения: PPO_S — политика, обученная с фильтром (базовая надлежащая реакция, ρ); PPO_N — обученная без фильтра; «+R» — исполнение с фильтром, «−R» — без; IDM_R — IDM/MOBIL + RSS без RL.

- **Семья A (основная)** — главный вопрос: насколько система «RL + RSS» отличается от «RL без фильтра» при каждом β. 6 сравнений, Холм по 6.
- **Семья B (деградация)** — растет ли риск RL + RSS с β: каждый β > 1 против β = 1 у той же системы. 5 сравнений, Холм по 5. По ней определяется β*.
- **Семья C (дополнительный γ)** — γ < 1 против γ = 1 при β = 1. 3 сравнения.

Остальное (перекрестная схема 2×2, IDM_R, задержка 0, аварийное торможение) — описательные сравнения без вывода о значимости.

In [3]:
pd.DataFrame([(fam, c) for fam, cs in COMPARISONS.items() for c in cs], columns=['семья', 'сравнение'])

,семья,сравнение
0,"A (основная, Холм по 6)",β = 1.0: PPO_S+R − PPO_N−R
1,"A (основная, Холм по 6)",β = 1.2: PPO_S+R − PPO_N−R
2,"A (основная, Холм по 6)",β = 1.4: PPO_S+R − PPO_N−R
3,"A (основная, Холм по 6)",β = 1.6: PPO_S+R − PPO_N−R
4,"A (основная, Холм по 6)",β = 1.8: PPO_S+R − PPO_N−R
5,"A (основная, Холм по 6)",β = 2.0: PPO_S+R − PPO_N−R
6,"B (деградация, Холм по 5)",PPO_S+R: β = 1.2 − β = 1
7,"B (деградация, Холм по 5)",PPO_S+R: β = 1.4 − β = 1
8,"B (деградация, Холм по 5)",PPO_S+R: β = 1.6 − β = 1
9,"B (деградация, Холм по 5)",PPO_S+R: β = 1.8 − β = 1


## 3. β* и верхняя точка сетки

β* — наименьший уровень сетки со знаком «хуже» в семье B, подтвержденный «хуже» на следующем уровне. Если «хуже» только на верхней точке β = 2, подтвердить нечем: исход «β* не подтвержден: хуже только на верхней точке сетки» (β* > 1,8 или вне сетки — неизвестно). Сетку ради подтверждения не расширяем.

In [4]:
L = [1.0, 1.2, 1.4, 1.6, 1.8, 2.0]
W, N = 'хуже', 'не различимо'
for signs in ([N, N, W, W, W, W], [N, W, N, W, W, N], [N, N, N, N, N, W], [N] * 6):
    print(signs[1:], '->', beta_star(L, signs)[1])

['не различимо', 'хуже', 'хуже', 'хуже', 'хуже'] -> хуже при β = 1.4, подтверждено при β = 1.6
['хуже', 'не различимо', 'хуже', 'хуже', 'не различимо'] -> хуже при β = 1.6, подтверждено при β = 1.8
['не различимо', 'не различимо', 'не различимо', 'не различимо', 'хуже'] -> β* не подтвержден: хуже только на верхней точке сетки β = 2
['не различимо', 'не различимо', 'не различимо', 'не различимо', 'не различимо'] -> β* не найден: ни на одном уровне сетки нет подтвержденного «хуже»


## 4. Перекрестный бутстреп и нулевые исходы

Исходы ячейки — матрица [политики × сцены]; сцены общие для всех систем и условий (одни seed сцен, одни расписания). В каждом повторе индексы сцен выбираются один раз и применяются ко всем сравниваемым матрицам; индексы политик — свои для разных систем и общие, если сравнивается одна и та же политика в двух условиях. Политики и сцены — перекрестная структура, а не независимые эпизоды в каждой ячейке.

Если в обеих ячейках 0 столкновений, бутстреп дал бы вырожденный интервал [0; 0] — это не доказательство отсутствия риска. Тогда для каждой системы отдельно — точная граница Клоппера — Пирсона (0 из n: 1 − 0,05^(1/n)). Она относится к одному варианту при независимых сценах, а не ко всем вариантам вместе.

In [5]:
pd.DataFrame({'сцен': [50, 100, 300, 500, 1000],
              'верхняя граница при 0 столкновений, %': [round(100 * zero_upper_bound(n), 2)
                                                     for n in [50, 100, 300, 500, 1000]]})

,сцен,"верхняя граница при 0 столкновений, %"
0,50,5.82
1,100,2.95
2,300,0.99
3,500,0.60
4,1000,0.30


**Демонстрация на пилоте 07** (IDLE вместо PPO, одна «политика», 50 сцен) — как работает правило на семье B для основного варианта фильтра. Это не результат исследования: IDLE не обученная политика.

In [6]:
g = pd.read_csv(RES / 'pilot_07' / 'outcomes.csv')
main = g[g['вариант'] == 'базовая, задержка ρ (основной)']
mat = {b: main[main.beta == b].sort_values('seed')['collision'].astype(float).to_numpy()[None, :]
       for b in sorted(main.beta.unique())}
rows = [compare(mat[b], mat[1.0], paired_policies=True, seed=1) | {'β': b} for b in sorted(mat) if b > 1]
res = pd.DataFrame(rows)
p = res['p'].fillna(1.0).to_numpy()
res['Холм: отвергнута'], res['Холм: уровень'] = holm(p)
res['ДИ, п.п.'] = [f'[{100*lo:.1f}; {100*hi:.1f}]' if np.isfinite(lo) else '—' for lo, hi in res['ci']]
res[['β', 'diff', 'ДИ, п.п.', 'p', 'Холм: уровень', 'Холм: отвергнута', 'sign', 'magnitude', 'note']].round(4)

,β,diff,"ДИ, п.п.",p,Холм: уровень,Холм: отвергнута,sign,magnitude,note
0,1.2,0.24,[12.0; 36.0],0.0,0.0100,True,хуже,за пределами допуска,
1,1.4,0.36,[22.0; 50.0],0.0,0.0125,True,хуже,за пределами допуска,
2,1.6,0.46,[32.0; 60.0],0.0,0.0167,True,хуже,за пределами допуска,
3,1.8,0.52,[38.0; 66.0],0.0,0.0250,True,хуже,за пределами допуска,
4,2.0,0.52,[38.0; 66.0],0.0,0.0500,True,хуже,за пределами допуска,


Для решения по сравнению i ДИ строится на уровне 1 − α_i шага Холма (колонка «Холм: уровень»); здесь для простоты показан 95 % ДИ.

## 5. Замер скорости PPO на целевой среде (бенчмарк пропускной способности)

**Это замер скорости, а не обучение.** Несколько тысяч шагов PPO (stable-baselines3) на целевой среде `TrainingEnv`: ShieldedVehicle, сцена контролируемого следования, номинальный режим, seed сцен от 1 000 000 (не пересекаются со сценами оценки). Отдельно для политики с базовым фильтром (базовая надлежащая реакция, ρ) и без фильтра (те же пределы машины: [−5; 3] м/с², рывок ≤ 10 м/с³), при 1 и 8 параллельных средах. Сохраняем только шагов в секунду и оценку часов на 10⁵ шагов на политику. Модели не сохраняются. Выводов о сходимости и о том, достаточно ли 10⁵ шагов, отсюда не делаем: бюджет обучения выбирается по кривым обучения пробных политик.

In [7]:
import subprocess, sys as _sys
BENCH_STEPS = 4096  # несколько тысяч шагов — только для замера скорости
# отдельным процессом: параллельные среды из ядра Jupyter на macOS зависают
subprocess.run([_sys.executable, '-m', 'rssrl.volume', str((PILOT / 'ppo_throughput.csv').resolve()), str(BENCH_STEPS)],
               cwd='..', check=True, capture_output=True)
bench = pd.read_csv(PILOT / 'ppo_throughput.csv')  # только скорость, моделей нет
bench.round(2)

,политика,сред,шагов в секунду,часов на 1e5 шагов на политику
0,с базовым фильтром,1,6.70,4.14
1,с базовым фильтром,8,26.26,1.06
2,без фильтра,1,6.68,4.16
3,без фильтра,8,24.78,1.12


Оценка часов — время на 10⁵ шагов при такой скорости, а не утверждение, что 10⁵ шагов достаточно.

## 6. Бюджет оценки по частям

Пропускная способность — из пилота 07 (эпизодов в час для всей параллельной пачки на этом компьютере; на число ядер ее больше не делим). Части считаются отдельно: основной β-эксперимент (6 уровней), дополнительный γ (4 уровня), контрфактические прогоны (на 100 сценах основного эксперимента, число прогонов на эпизод — по пилоту 06; у PPO оно может быть другим). Варианты с фильтром — только при основной задержке ρ. Время PPO-эпизода не замерено: берем время эпизода IDLE, это оценка снизу (политике нужен еще прямой проход сети).

In [8]:
timing = json.loads((RES / 'pilot_07' / 'timing.json').read_text())
eph = timing['episodes_per_hour']
print(f"пилот 07: {eph:.0f} эпизодов в час на {timing['cpu']} ядрах (средняя длина эпизода "
      f"{timing['mean_episode_sim_s']:.0f} с, загрузка {timing['load_1min']:.1f})")
cf = pd.read_csv(RES / 'pilot_06' / 'counterfactual.csv')
main_cf = cf[(cf['сценарий'] == 'тормозящий лидер, IDLE') & (cf.reaction_delay == 0.5)]
CF_PER_EP = len(main_cf) / (20 * main_cf.level.nunique())  # вмешательств на эпизод в пилоте 06
print(f'контрфактических прогонов на эпизод (пилот 06, тормозящий лидер, ρ): {CF_PER_EP:.1f}')
rows = []
for k in (3, 5):
    for n in (300, 500, 1000):
        b = budget(n_scenes=n, n_policies=k, cf_per_episode=CF_PER_EP)
        rows.append({'политик на тип': k, 'сцен': n,
                     'эпизодов на сцену и уровень': b['эпизодов на сцену и уровень'],
                     'основной β, ч': round(b['основной β'] / eph, 1),
                     'доп. γ, ч': round(b['дополнительный γ'] / eph, 1),
                     'контрфакт, ч': round(b['контрфакт'] / eph, 1)})
pd.DataFrame(rows).set_index(['политик на тип', 'сцен'])

пилот 07: 2135 эпизодов в час на 8 ядрах (средняя длина эпизода 34 с, загрузка 72.9)
контрфактических прогонов на эпизод (пилот 06, тормозящий лидер, ρ): 6.5


эпизодов на сцену и уровень  основной β, ч  доп. γ, ч  \
политик на тип сцен                                                          
3              300                            13           11.0        7.3   
               500                            13           18.3       12.2   
               1000                           13           36.5       24.4   
5              300                            21           17.7       11.8   
               500                            21           29.5       19.7   
               1000                           21           59.0       39.4   

                     контрфакт, ч  
политик на тип сцен                
3              300           12.9  
               500           12.9  
               1000          12.9  
5              300           20.3  
               500           20.3  
               1000          20.3

## Выводы этапа 8

1. **Правило решения** — две независимые характеристики (знак и величина относительно ±2 п.п.); каждый интервал получает ровно одну пару меток.
2. **Основные сравнения**: семья A (RL + RSS против RL без фильтра при каждом β, Холм по 6), семья B (деградация RL + RSS: β против β = 1, Холм по 5, по ней β*), семья C (γ против γ = 1, Холм по 3). β* при «хуже» только на верхней точке сетки — «не подтвержден».
3. **Бутстреп** — перекрестный, с общими индексами сцен для всех систем и условий; при нуле столкновений — точная граница Клоппера — Пирсона, а не [0; 0].
4. **Замер скорости PPO** (не обучение, моделей нет): ≈ 6,7 шага/с на 1 среде и ≈ 25–26 шагов/с на 8 средах, с фильтром и без почти одинаково; при такой скорости 10⁵ шагов — около 1,1 ч на политику на 8 средах. О сходимости и достаточности 10⁵ шагов отсюда ничего не следует.
5. **Бюджет по частям** при 3 политиках на тип и 500 сценах: основной β ≈ 18 ч, дополнительный γ ≈ 12 ч, контрфакт ≈ 13 ч (оценки снизу: время эпизода PPO не замерено). Окончательный объем — после 2–3 пробных политик: `projected_halfwidth` на их исходах для семьи A.